In [ ]:
import pandas as pd
import pyarrow
import sys

df_raw = pd.read_json("../data/tft_matches.json")

df_fact = df_raw[["match_id","player","placement","level"]].copy()

df_traits = df_raw[["player","traits"]].explode("traits")
df_traits = df_traits.rename(columns={"traits": "trait"})

df_units = df_raw[["player","units"]].explode("units")
df_units = df_units.rename(columns={"units": "unit"})

fact_traits = df_fact.merge(df_traits, on="player", how="left")
fact_units = df_fact.merge(df_units, on="player", how="left")

print(len(df_raw))
print(len(df_fact))
print(len(fact_traits))

fact_traits.groupby("trait")["placement"].mean().sort_values()
fact_units.groupby("unit")["placement"].mean().sort_values()


df_fact.to_parquet("../data/fact.parquet", engine="pyarrow", index=False)
fact_traits.to_parquet("../data/fact_traits.parquet", index=False)
fact_units.to_parquet("../data/fact_units.parquet", index=False)

df_fact = pd.read_parquet("../data/fact.parquet")
df_fact.head()


NameError: name 'run_pipeline' is not defined

In [5]:
import pandas as pd
import logging

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

def validate_columns(df):
    required_columns = [
        "match_id",
        "player",
        "placement",
        "level",
        "traits",
        "units"
    ]

    missing = [
        col for col in required_columns
        if col not in df.columns
    ]

    if missing:
        raise ValueError(f"Missing columns: {missing}")
    
def validate_nulls(df):
    if df["player"].isnull().any():
        raise ValueError("Null values found in 'player' column")
    if df["match_id"].isnull().any():
        raise ValueError("Null values found in 'match_id' column")
    if df["placement"].isnull().any():
        raise ValueError("Null values found in 'placement' column")
    
def validate_placement(df):
    if not df["placement"].between(1, 8).all():
        raise ValueError("Invalid values found in 'placement' column")
    
def validate_duplicates(df):
    duplicates = df.duplicated(subset=["match_id", "player"])
    if duplicates.any():
        raise ValueError("Duplicate rows found based on 'match_id' and 'player'")
    
def validate_data(df):
    validate_columns(df)
    validate_nulls(df)
    validate_placement(df)
    validate_duplicates(df)

    logging.info("Data validation passed successfully.")

def run_pipeline():

    df =  pd.read_json("../data/tft_matches.json")
    
    validate_data(df)
    
    df_fact, fact_traits, fact_units = transform(df)

    load(df_fact, fact_traits, fact_units)

    logging.info("Pipeline executed successfully.")

def load(df_fact,fact_traits,fact_units):

    df_fact.to_parquet("../data/fact.parquet", engine="pyarrow", index=False)
    fact_traits.to_parquet("../data/fact_traits.parquet", index=False)
    fact_units.to_parquet("../data/fact_units.parquet", index=False)

    logging.info("Data loaded successfully.")

def transform(df):

    df_fact = df[["match_id","player","placement","level"]].copy()

    df_traits = df[["player","traits"]].explode("traits")
    df_traits = df_traits.rename(columns={"traits": "trait"})

    df_units = df[["player","units"]].explode("units")
    df_units = df_units.rename(columns={"units": "unit"})

    fact_traits = df_fact.merge(df_traits, on="player", how="left")
    fact_units = df_fact.merge(df_units, on="player", how="left")

    logging.info("Data transformation completed successfully.")

    return df_fact, fact_traits, fact_units

try:
    logging.info("Starting pipeline execution.")
    run_pipeline()
    logging.info("Pipeline execution completed successfully.")

except Exception as e:
    logging.error(f"Pipeline execution failed: {e}")

2026-05-24 01:51:34,816 - INFO - Starting pipeline execution.
2026-05-24 01:51:34,819 - INFO - Data validation passed successfully.
2026-05-24 01:51:34,826 - INFO - Data transformation completed successfully.
2026-05-24 01:51:34,830 - INFO - Data loaded successfully.
2026-05-24 01:51:34,830 - INFO - Pipeline executed successfully.
2026-05-24 01:51:34,831 - INFO - Pipeline execution completed successfully.


In [14]:
import sys
import os


sys.path.append(
    os.path.abspath("..")
)

from src.pipeline import run_pipeline

run_pipeline()






TypeError: 'module' object is not callable. Did you mean: 'transform.transform(...)'?